In [1]:
import pandas as pd
import numpy as np
import time
import sys
sys.path.append('..')
from preprocessing import *
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from cuml.ensemble import RandomForestRegressor as cuRF
print('All imports OK')

All imports OK


In [2]:
df = pd.read_csv(r'../output_csv/ANL-Intrepid-2009-1.swf.csv')
print(f'Dataset shape: {df.shape}')

Dataset shape: (68936, 18)


In [3]:
feature_columns = ['requested_processors', 'requested_time', 'submit_time', 'wait_time', 'user_id', 'queue_id']
target_column = 'run_time'

X_train, X_test, Y_train, Y_test, scaler = prepare_data(df, feature_columns, target_column, statuss=-1)
print(f'Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}')
print(f'Features: {feature_columns}')

Train size: 40374, Test size: 10094
Features: ['requested_processors', 'requested_time', 'submit_time', 'wait_time', 'user_id', 'queue_id']


In [4]:
# Train Random Forest (GPU - cuML)
rmse_lst, mae_lst, r2_lst, infer_time_lst = [], [], [], []

for iter in range(5):
    print(f"Iteration {iter + 1} / 5")
    
    # Re-split data each iteration for robust evaluation
    X_train, X_test, Y_train, Y_test, scaler = prepare_data(df, feature_columns, target_column, statuss=-1)
    
    # Train cuML Random Forest on GPU
    rf_model = cuRF(
        n_estimators=100,
        max_depth=16,
        min_samples_split=2,
        max_features=1.0,
        bootstrap=True
    )
    rf_model.fit(X_train, Y_train)
    
    # Predict & measure inference time
    start_time = time.time()
    Y_pred_scaled = rf_model.predict(X_test)
    end_time = time.time()
    
    # Inverse transform to get original scale
    n_features = len(feature_columns)
    dummy = np.zeros((len(Y_pred_scaled), n_features + 1))
    dummy[:, -1] = Y_pred_scaled
    Y_pred = scaler.inverse_transform(dummy)[:, -1]
    
    dummy_true = np.zeros((len(Y_test), n_features + 1))
    dummy_true[:, -1] = Y_test
    Y_true = scaler.inverse_transform(dummy_true)[:, -1]
    
    rmse = np.sqrt(mean_squared_error(Y_true, Y_pred))
    mae = mean_absolute_error(Y_true, Y_pred)
    r2 = r2_score(Y_true, Y_pred)
    infer_time = (end_time - start_time) / len(Y_test)
    
    rmse_lst.append(rmse)
    mae_lst.append(mae)
    r2_lst.append(r2)
    infer_time_lst.append(infer_time)

Iteration 1 / 5
Iteration 2 / 5
Iteration 3 / 5
Iteration 4 / 5
Iteration 5 / 5


In [5]:
# Save results
results_df = pd.DataFrame({
    'RMSE': rmse_lst,
    'MAE': mae_lst,
    'R2': r2_lst,
    'Inference Time': infer_time_lst
})
results_df.to_csv('../output_ANL/RandomForest_results.csv', index=False)

print(f"\n{'='*60}")
print(f"Random Forest (GPU - cuML) — ANL Dataset")
print(f"{'='*60}")
print(f"Mean RMSE: {np.mean(rmse_lst):.4f} ± {np.std(rmse_lst):.4f}")
print(f"Mean MAE:  {np.mean(mae_lst):.4f} ± {np.std(mae_lst):.4f}")
print(f"Mean R²:   {np.mean(r2_lst):.4f} ± {np.std(r2_lst):.4f}")
print(f"Mean Inference Time: {np.mean(infer_time_lst):.6f}s")


Random Forest (GPU - cuML) — ANL Dataset
Mean RMSE: 5949.8107 ± 785.5018
Mean MAE:  1347.5920 ± 44.6794
Mean R²:   0.6491 ± 0.0489
Mean Inference Time: 0.000032s
